# 实验 1：Grad-CAM 模型可视化

使用 ImageNet 预训练 ResNet-18，对 CIFAR-10 测试集中的 5 个类别各选 1 张图，生成原图、Grad-CAM 热力图和叠加图。数据、模型缓存和结果均保存在当前目录下的 `data/`、`models/`、`results/`。首次运行需要联网下载数据和模型权重。


In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torchvision.datasets import CIFAR10
from torchvision.models import resnet18, ResNet18_Weights

ROOT = Path.cwd()
if not (ROOT / 'exp3A.ipynb').exists() and (ROOT / 'exp3' / 'exp3A.ipynb').exists():
    ROOT = ROOT / 'exp3'
DATA_DIR = ROOT / 'data'
RESULT_DIR = ROOT / 'results'
RESULT_DIR.mkdir(exist_ok=True)
torch.hub.set_dir(str(ROOT / 'models'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
model = resnet18(weights=weights).to(device).eval()
imagenet_labels = weights.meta['categories']
dataset = CIFAR10(root=str(DATA_DIR), train=False, download=True)
print('device:', device, '| CIFAR-10 测试集:', len(dataset))

## Grad-CAM 计算

目标层取 ResNet-18 的最后一个残差块 `layer4[-1].conv2`。对预测类别分数反向传播，将目标层梯度在空间维度做全局平均池化得到通道权重，再对特征图加权求和、经过 ReLU 并上采样到输入尺寸。

In [ ]:
target_layer = model.layer4[-1].conv2
cache = {}

def save_activation(module, inputs, output):
    cache['activation'] = output
    output.register_hook(lambda grad: cache.__setitem__('gradient', grad))

hook = target_layer.register_forward_hook(save_activation)

def gradcam(image_tensor):
    x = preprocess(image_tensor).unsqueeze(0).to(device)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    pred_id = int(logits.argmax(dim=1).item())
    score = logits[0, pred_id]
    score.backward()

    act = cache['activation'][0]
    grad = cache['gradient'][0]
    channel_weights = grad.mean(dim=(1, 2))
    cam = torch.relu((channel_weights[:, None, None] * act).sum(dim=0))
    cam = cam[None, None]
    cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[0, 0]
    cam = cam.detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    confidence = float(logits.softmax(dim=1)[0, pred_id].detach().item())
    return pred_id, confidence, cam

# 取 CIFAR-10 的前五类：飞机、汽车、鸟、猫、鹿；每类取测试集第一张图片。
class_names = dataset.classes
selected = {}
for i, label in enumerate(dataset.targets):
    if label < 5 and label not in selected:
        selected[label] = i
    if len(selected) == 5:
        break

for true_id in range(5):
    image, _ = dataset[selected[true_id]]
    pred_id, confidence, cam = gradcam(image)
    original = image.resize((224, 224))
    rgb = np.asarray(original) / 255.0

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
    axes[0].imshow(rgb)
    axes[0].set_title(f'原图\n真实类别：{class_names[true_id]}')
    axes[1].imshow(cam, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title('Grad-CAM 热力图')
    axes[2].imshow(rgb)
    axes[2].imshow(cam, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[2].set_title(f'叠加图\n预测：{imagenet_labels[pred_id]} ({confidence:.1%})')
    for ax in axes:
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(RESULT_DIR / f'gradcam_{true_id}_{class_names[true_id]}.png', dpi=160)
    plt.show()
    print(f'真实类别：{class_names[true_id]} | 预测类别：{imagenet_labels[pred_id]} | 置信度：{confidence:.3f}')

hook.remove()
print('结果图片已保存到：', RESULT_DIR)

## 结果记录

运行后结合五组叠加图逐图填写：热力区域是否落在目标物体上、是否存在背景误关注，以及预测类别和真实类别是否一致。Grad-CAM 仅显示该次预测所依赖的主要图像区域，不能单独证明模型整体正确或具有因果解释能力。